#### RFM ANALYSIS
- Recency(R) - How long has it been since the customer's last purchase or interaction
- Frequency(F) - How often do they use the service or buy products
- Monetory(M) - How much total revenue has the customer generated

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors

In [27]:
data = pd.read_csv('../data/enriched/telco_churn_enriched.csv')

In [28]:
data.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [29]:
data.tail().T

,7038,7039,7040,7041,7042
customerID,6840-RESVB,2234-XADUH,4801-JZAZL,8361-LTMKD,3186-AJIEK
gender,Male,Female,Female,Male,Male
SeniorCitizen,0,0,0,1,0
Partner,Yes,Yes,Yes,Yes,No
Dependents,Yes,Yes,Yes,No,No
tenure,24,72,11,4,66
PhoneService,Yes,Yes,No,Yes,Yes
MultipleLines,Yes,Yes,No phone service,Yes,No
InternetService,DSL,Fiber optic,DSL,Fiber optic,Fiber optic
OnlineSecurity,Yes,No,Yes,No,Yes


In [30]:
data.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
Total_Services       0
Churn_Numeric        0
dtype: int64

In [31]:
data['TotalCharges'] = data['TotalCharges'].fillna(data['MonthlyCharges'])

In [38]:
# R - tenure
# F - TotalServices
# M - TotalCharges
# rfm_data = data[['customerID', 'tenure', 'Total_Services', 'TotalCharges']]
rfm_data = data.groupby('customerID').agg({
    'tenure': 'max',            # Recency (High tenure = High loyalty)
    'Total_Services': 'sum',    # Frequency (Number of active services)
    'TotalCharges': 'sum'       # Monetary (Lifetime Value)
})

In [39]:
rfm_data.rename(columns={
    'tenure': 'Recency',
    'Total_Services': 'Frequency',
    'TotalCharges': 'Monetary'
}, inplace=True)

In [40]:
rfm_data.head()

,Recency,Frequency,Monetary
customerID,,,
0002-ORFBO,9,3,593.30
0003-MKNFE,9,1,542.40
0004-TLHLJ,4,1,280.85
0011-IGKFF,13,4,1237.85
0013-EXCHZ,3,2,267.40


In [41]:
# Define quantiles
quantiles = rfm_data.quantile(q=[0.25, 0.5, 0.75])

# Assign R, F, M scores
def RScore(x,p,d):
    if p == 'Recency':
        if x <= d[p][0.25]:
            return 4
        elif x <= d[p][0.50]:
            return 3
        elif x <= d[p][0.75]:
            return 2
        else:
            return 1
    else:
        if x <= d[p][0.25]:
            return 1
        elif x <= d[p][0.50]:
            return 2
        elif x <= d[p][0.75]:
            return 3
        else:
            return 4
        
rfm_data['R'] = rfm_data['Recency'].apply(RScore, args=('Recency', quantiles))
rfm_data['F'] = rfm_data['Frequency'].apply(RScore, args=('Frequency', quantiles))
rfm_data['M'] = rfm_data['Monetary'].apply(RScore, args=('Monetary', quantiles))

rfm_data.head()

,Recency,Frequency,Monetary,R,F,M
customerID,,,,,,
0002-ORFBO,9,3,593.30,4,3,2
0003-MKNFE,9,1,542.40,4,2,2
0004-TLHLJ,4,1,280.85,4,2,1
0011-IGKFF,13,4,1237.85,3,4,2
0013-EXCHZ,3,2,267.40,4,2,1


In [42]:
rfm_data['RFM_Segment'] = rfm_data['R'].astype(str) + rfm_data['F'].astype(str) + rfm_data['M'].astype(str)
rfm_data['RFM_Score'] = rfm_data[['R', 'F', 'M']].sum(axis=1)
rfm_data.head()

,Recency,Frequency,Monetary,R,F,M,RFM_Segment,RFM_Score
customerID,,,,,,,,
0002-ORFBO,9,3,593.30,4,3,2,432,9
0003-MKNFE,9,1,542.40,4,2,2,422,8
0004-TLHLJ,4,1,280.85,4,2,1,421,7
0011-IGKFF,13,4,1237.85,3,4,2,342,9
0013-EXCHZ,3,2,267.40,4,2,1,421,7


In [43]:
rfm_data.RFM_Score.unique()

array([ 9,  8,  7,  6,  5, 10,  4])

In [50]:
segment_labels = ['Low-Value', 'Mid-Value', 'High-Value']

def assign_segment(score):
    if score < 6:
        return 'Low-Value'
    elif score < 8:
        return 'Mid-Value'
    else:
        return 'High-Value'

rfm_data['RFM_Segment_Label'] = rfm_data['RFM_Score'].apply(assign_segment)
rfm_data.head()

,Recency,Frequency,Monetary,R,F,M,RFM_Segment,RFM_Score,RFM_Segment_Label
customerID,,,,,,,,,
0002-ORFBO,9,3,593.30,4,3,2,432,9,High-Value
0003-MKNFE,9,1,542.40,4,2,2,422,8,High-Value
0004-TLHLJ,4,1,280.85,4,2,1,421,7,Mid-Value
0011-IGKFF,13,4,1237.85,3,4,2,342,9,High-Value
0013-EXCHZ,3,2,267.40,4,2,1,421,7,Mid-Value


In [43]:
segment_counts = rfm_data['RFM_Segment_Label'].value_counts().reset_index()
segment_counts.columns = ['RFM_Segment', 'Count']
segment_counts = segment_counts.sort_values('RFM_Segment')

In [52]:
# Create a bar plot
fig = px.bar(
    segment_counts,
    x='RFM_Segment',
    y='Count',
    title='Customer Distribution by RFM Segment',
    labels={'RFM_Segment': 'RFM Segment', 'Count': 'Number of Customers'},
    color='RFM_Segment',
    color_discrete_sequence=plotly.colors.qualitative.Pastel
)

fig.show()

In [71]:
rfm_data['RFM_Customer_Segment'] = ''
rfm_data.loc[rfm_data['RFM_Score'] >= 9, 'RFM_Customer_Segment'] = 'VIP/Loyal'
rfm_data.loc[(rfm_data['RFM_Score'] >= 7) & (rfm_data['RFM_Score'] < 9), 'RFM_Customer_Segment'] = 'Potential Loyal'
rfm_data.loc[(rfm_data['RFM_Score'] >= 6) & (rfm_data['RFM_Score'] < 7), 'RFM_Customer_Segment'] = 'At Risk Customers'
rfm_data.loc[(rfm_data['RFM_Score'] >= 5) & (rfm_data['RFM_Score'] < 6), 'RFM_Customer_Segment'] = 'Hibernating'
rfm_data.loc[(rfm_data['RFM_Score'] >= 4) & (rfm_data['RFM_Score'] < 5), 'RFM_Customer_Segment'] = 'Lost'
segment_counts = rfm_data['RFM_Customer_Segment'].value_counts().reset_index()

In [72]:
segment_product_counts = rfm_data.groupby(['RFM_Segment_Label', 'RFM_Customer_Segment']).size().reset_index(name='Count')
segment_product_counts = segment_product_counts.sort_values('Count', ascending=False)

In [73]:
# Create a tree map
fig_treemap_segment_product = px.treemap(
    segment_product_counts,
    path=['RFM_Segment_Label', 'RFM_Customer_Segment'],
    values='Count',
    color='RFM_Segment_Label',
    color_discrete_sequence=plotly.colors.qualitative.Pastel,
    title = 'RFM Customer Segments by value'
)
fig_treemap_segment_product.show()

In [74]:
vip_segment = rfm_data[rfm_data['RFM_Customer_Segment'] == 'VIP/Loyal']

In [46]:
import plotly.graph_objects as go
import plotly.colors

# 1. Prepare the data and EXPLICITLY rename columns to avoid KeyErrors
segment_counts = rfm_data['RFM_Customer_Segment'].value_counts().reset_index()
segment_counts.columns = ['RFM_Customer_Segment', 'Count'] # Force names to be consistent

# 2. Define your colors
pastel_colors = plotly.colors.qualitative.Pastel
vip_color = 'rgb(158,202,225)'

# 3. Initialize the figure
fig = go.Figure(
    data=[
        go.Bar(
            x=segment_counts['RFM_Customer_Segment'], 
            y=segment_counts['Count'], # Now matches the explicit name above
            marker=dict(
                color=[vip_color if segment == 'VIP/Loyal' else pastel_colors[i % len(pastel_colors)] 
                       for i, segment in enumerate(segment_counts['RFM_Customer_Segment'])]
            ),
            marker_line_color='rgb(8,48,107)',
            marker_line_width=1.5, 
            opacity=0.6
        )
    ]
)

# 4. Update the layout
fig.update_layout(
    title='Comparison of RFM Segments',
    xaxis_title='RFM Segment',
    yaxis_title='Number of Customers',
    showlegend=False,
    template='plotly_white'
)

fig.show()

In [78]:
correlation_matrix = vip_segment[['R', 'F', 'M']].corr()

In [79]:
fig_heatmap = go.Figure(
    data=go.Heatmap(
        z = correlation_matrix.values,
        x = correlation_matrix.columns,
        y = correlation_matrix.columns,
        colorscale='RdBu',
        colorbar=dict(title='Correlation Coefficient')
    )
)

fig_heatmap.update_layout(title='Correlation Matrix for VIP/Loyal Customers')
fig_heatmap.show()

In [48]:
import plotly.graph_objects as go
import plotly.colors

# 1. Prepare the data and EXPLICITLY rename columns to avoid KeyErrors
segment_counts = rfm_data['RFM_Customer_Segment'].value_counts().reset_index()
segment_counts.columns = ['RFM_Customer_Segment', 'Count'] # Force names to be consistent

# 2. Define your colors
pastel_colors = plotly.colors.qualitative.Pastel
vip_color = 'rgb(158,202,225)'

# 3. Initialize the figure
fig = go.Figure(
    data=[
        go.Bar(
            x=segment_counts['RFM_Customer_Segment'], 
            y=segment_counts['Count'], # Now matches the explicit name above
            marker=dict(
                color=[vip_color if segment == 'VIP/Loyal' else pastel_colors[i % len(pastel_colors)] 
                       for i, segment in enumerate(segment_counts['RFM_Customer_Segment'])]
            ),
            marker_line_color='rgb(8,48,107)',
            marker_line_width=1.5, 
            opacity=0.6
        )
    ]
)

# 4. Update the layout
fig.update_layout(
    title='Comparison of RFM Segments',
    xaxis_title='RFM Segment',
    yaxis_title='Number of Customers',
    showlegend=False,
    template='plotly_white'
)

fig.write_image('../outputs/figures/rfm_segment_comparison.png')
fig.show()

In [49]:
# Group by RFM segments and calculate mean scores
segment_scores = rfm_data.groupby('RFM_Customer_Segment')[['R', 'F', 'M']].mean().reset_index()

# Create figure
fig = go.Figure()

# Add bars for Recency score
fig.add_trace(go.Bar(
    x=segment_scores['RFM_Customer_Segment'],
    y=segment_scores['R'],
    name='Recency Score',
    marker_color='rgb(158,202,225)'
))

# Add bars for Frequency score
fig.add_trace(go.Bar(
    x=segment_scores['RFM_Customer_Segment'],
    y=segment_scores['F'],
    name='Frequency Score',
    marker_color='rgb(94,158,217)'
))

# Add bars for Monetary score
fig.add_trace(go.Bar(
    x=segment_scores['RFM_Customer_Segment'],
    y=segment_scores['M'],
    name='Monetary Score',
    marker_color='rgb(32,102,148)'
))

# Update layout
fig.update_layout(
    title='Comparison of RFM Segments based on Recency, Frequency, and Monetary Scores',
    xaxis_title='RFM Segments',
    yaxis_title='Score',
    barmode='group',
    showlegend=True
)


fig.write_image('../outputs/figures/rfm_segment_score_comparison.png')
# Display the figure
fig.show()

🎯 Project Overview
This analysis integrates Machine Learning (Logistic Regression) with Behavioral Segmentation (RFM) to provide a 360-degree view of customer retention. By identifying not just who is likely to churn, but the financial value at risk, we move from reactive data science to proactive business strategy.

🔍 1. Key Predictive Insights (The "Why")
Our model identifies the specific features that drive customers toward or away from leaving the company:

- Top Churn Drivers (Risk Factors):

- Internet Service (Fiber Optic): This is the strongest predictor of churn, suggesting potential pricing or reliability issues.

- Payment Method (Electronic Check): Customers using this manual method are significantly more likely to leave than those on automated plans.

- Top Retention Anchors (Loyalty Drivers):

- Two-Year Contracts: The most powerful deterrent to churn.

- Tenure: Long-term relationship length is a massive stabilizer for the customer base.

💎 2. Customer Value Segmentation (The "Who")
We categorized the 7,043 customers into five actionable value tiers based on their Recency, Frequency, and Monetary scores:

- VIP/Loyal (2,015 Customers): High-intensity users with the highest Monetary value, often exceeding $8,000 in lifetime charges.

- Potential Loyal (2,806 Customers): Our largest and most stable group, characterized by high Tenure but lower service frequency.

- At Risk & Hibernating: Customers showing declining engagement scores who represent the primary "battleground" for retention efforts.

🚀 3. Strategic Roadmap for Retention
Based on the intersection of Risk and Value, the following actions are recommended:

- Lock in the VIPs: Proactively offer Two-Year Contract incentives to the VIP/Loyal segment to protect the highest-revenue accounts.

- Automate Payments: Launch a campaign to transition At Risk customers from Electronic Checks to Credit Card (Automatic) to remove friction from the billing cycle.

- Audit Fiber Optic: Investigate the service quality for Fiber Optic users, as this high-speed tier is currently our most vulnerable segment.